In [1]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

### Raw Data Loading

In [29]:
import requests

from product_dto import ProductDto, Material, Price

In [35]:
def fetch_data_from_url(url: str,
                        headers: dict[str, str] | None = None,
                        params: dict[str, str] | None = None) -> dict:
    requests_args = {
        "url": url, "verify": False
    }
    if headers is not None:
        requests_args["headers"] = headers
    if params is not None:
        requests_args["params"] = params

    response = requests.get(**requests_args)

    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Failed to fetch data: {response.status_code}")
        raise Exception(response.text)

In [36]:
def fetch_all_products(page_size, page_number) -> list:
    url = "https://api.hm.com/search-services/v1/en_gb/listing/resultpage"

    params_ladies = {
        "page": page_number,
        "page-size": page_size,
        "pageId": "/ladies/shop-by-product/view-all",
        "sort": "RELEVANCE",
        "categoryId": "ladies_all",
        "facets": "",
        "rFacets": "",
        "skipStockCheck": "false",
        "pageSource": "PLP",
        "touchPoint": "android"
    }

    ladies_response = fetch_data_from_url(url, params=params_ladies)
    ladies_products = ladies_response["plpList"]["productList"]

    params_men = params_ladies.copy()
    params_men["categoryId"] = "men_viewall"
    params_men["pageId"] = "/men/shop-by-product/view-all"

    men_response = fetch_data_from_url(url, params=params_men)
    men_products = men_response["plpList"]["productList"]

    all_products = ladies_products + men_products

    return all_products

In [37]:
def fetch_product_details(id) -> dict:
    url = f"https://app2.hm.com/hmwebservices/service/article/get-article-by-code/hm-greatbritain/Online/{id}/en.json"
    headers = {
        'Host': 'app2.hm.com',
        'user-agent': 'targetapp_android_22',
        'accept': 'application/json',
        'content-type': 'application/json',
        'x-os-version': '13',
        'x-app-version': '25.44.0',
        'Cookie': 'ak_bmsc=E76EF967F40F536F065E44242BE7AA1D~000000000000000000000000000000~YAAQSc4uFwzfvMuZAQAAzhYb6B2nbtJBZwUO+39Yd0MzjkOAmi1HNhv31NbqMCXpspl0W/1hVGZkp/r419qoNF+vYOSgCr54jS2AbrUGWHPSrjHjl8FwaD5dPqPiiolYWSdx2NFdbwdWap1lJSQfuDodS1KJ9+YxBZfgMC1DXqM9XIHHdO50EDz+Y5YezMSf2mPJ17BbNcH/smYpAMLZyicb1id24+k6j1TGvoiz4xWdDcvHFHxBtcOTacQtdviP1/7mlQy9DApd/FMPMBTs6u4xdBoqn9dQYdjnXZokxuDngqLjl3qoUHjCJRqCR/2nTeQGdK5JIKbhnWGsX1hHkTUsMPIIPa4rpA==; akainst=EU1; akamref=; redirectD4M=false'
    }
    return fetch_data_from_url(url, headers=headers)

In [38]:
PAGES = 12
PAGE_SIZE = 36  # all records: 864 (men+women)

products = []

for page_indx in range(1, PAGES+1):
    print(f"Proccessing page {page_indx}/{PAGES}")
    page_data = fetch_all_products(PAGE_SIZE, page_indx)

    for product in page_data:
        product_details = fetch_product_details(product.get("id"))

        materials = []
        for article in product_details.get("product").get("articlesList"):
            if article.get("code") == product_details.get("product").get("code"):
                materials = [
                    Material(m.get("name"), m.get("percentage"))
                    for m in article.get("compositions")[0].get("materials")
                ]
                break

        product_dto = ProductDto(
            name=product.get("productName"),
            description=product_details.get("product").get("description"),
            color=product_details.get("product").get("color").get("text"),
            construction=product_details.get("product").get("constructionDescr"),
            materials=materials,
            price=Price(
                value=product_details.get("product").get("whitePrice").get("price"),
                currency=product_details.get("product").get("whitePrice").get("currency")
            ),
            sizes=[s.get("label") for s in product.get("sizes")],
            sex=product_details.get("product").get("customerGroup"),
            image_url=product.get("modelImage")
        )
        products.append(product_dto)

Proccessing page 1/12
Proccessing page 2/12
Proccessing page 3/12
Proccessing page 4/12
Proccessing page 5/12
Proccessing page 6/12
Proccessing page 7/12
Proccessing page 8/12
Proccessing page 9/12
Proccessing page 10/12
Proccessing page 11/12
Proccessing page 12/12


In [46]:
import pandas as pd

products_df = pd.DataFrame(products)
products_df.head()

,name,description,color,construction,materials,price,sizes,sex,image_url
0,Tailored drawstring trousers,Loose-fit trousers in woven fabric featuring a...,Mole,Woven,"[{'name': 'Elastane', 'percentage': '4.00'}, {...","{'value': 19.99, 'currency': 'GBP'}","[22, 6, 8, 4, 14, 16, 10, 12, 18, 20]",Women,https://image.hm.com/assets/hm/6d/cf/6dcf62d65...
1,Barrel Leg High Waist Jeans,5-pocket jeans in rigid cotton denim that feel...,Washed black,Woven,"[{'name': 'Cotton', 'percentage': '100.00'}]","{'value': 24.99, 'currency': 'GBP'}","[4, 22, 10, 12, 6, 8, 18, 20, 14, 16]",Women,https://image.hm.com/assets/hm/77/f7/77f7d75e9...
2,Jacquard-knit jacket,"Jacket in a jacquard knit with a high, folded ...",Light beige/Checked,Knitted,"[{'name': 'Polyamide', 'percentage': '30.00'},...","{'value': 39.99, 'currency': 'GBP'}","[M, S, XS]",Women,https://image.hm.com/assets/hm/96/da/96daaeeb8...
3,Cotton poplin shirt,Long-sleeved shirt in crisp cotton poplin with...,Blue/Striped,Woven,"[{'name': 'Cotton', 'percentage': '100.00'}]","{'value': 22.99, 'currency': 'GBP'}","[M, L, XS, S, XL, XXL, XXS]",Women,https://image.hm.com/assets/hm/9c/40/9c405e26c...
4,Scuba sweatshirt,Loose-fit sweatshirt in scuba fabric with a st...,Grey marl,Knitted,"[{'name': 'Elastane', 'percentage': '8.00'}, {...","{'value': 27.99, 'currency': 'GBP'}","[XS, M, S, L]",Women,https://image.hm.com/assets/hm/81/18/81186b3f2...


### Get unique values of data

In [56]:
import json
import os

construction_uniques = products_df['construction'].unique()
materials_uniques = products_df['materials'].explode().apply(lambda x: x.get('name')).unique()
sizes_uniques = products_df['sizes'].explode().unique()
sex_uniques = products_df['sex'].unique()
currency_uniques = products_df["price"].apply(lambda x: x.get('currency')).unique()

### Transform Data

In [48]:
print(currency_uniques)

['GBP']


In [49]:
"""Ideally fetched from the NBP API; hard-coded here for simplicity."""
to_pln_factor = {
    'GBP': 4.85
}

In [50]:
def convert_price_to_pln(price_obj, to_pln_factor):
    if price_obj.get('currency') == 'PLN':
        return price_obj.get('value')
    elif price_obj.get('currency') in to_pln_factor:
        return round(price_obj.get('value') * to_pln_factor[price_obj.get('currency')], 2)
    else:
        return None

products_df['price_pln'] = products_df['price'].apply(lambda x: convert_price_to_pln(x, to_pln_factor))

In [51]:
products_df.head()

,name,description,color,construction,materials,price,sizes,sex,image_url,price_pln
0,Tailored drawstring trousers,Loose-fit trousers in woven fabric featuring a...,Mole,Woven,"[{'name': 'Elastane', 'percentage': '4.00'}, {...","{'value': 19.99, 'currency': 'GBP'}","[22, 6, 8, 4, 14, 16, 10, 12, 18, 20]",Women,https://image.hm.com/assets/hm/6d/cf/6dcf62d65...,96.95
1,Barrel Leg High Waist Jeans,5-pocket jeans in rigid cotton denim that feel...,Washed black,Woven,"[{'name': 'Cotton', 'percentage': '100.00'}]","{'value': 24.99, 'currency': 'GBP'}","[4, 22, 10, 12, 6, 8, 18, 20, 14, 16]",Women,https://image.hm.com/assets/hm/77/f7/77f7d75e9...,121.20
2,Jacquard-knit jacket,"Jacket in a jacquard knit with a high, folded ...",Light beige/Checked,Knitted,"[{'name': 'Polyamide', 'percentage': '30.00'},...","{'value': 39.99, 'currency': 'GBP'}","[M, S, XS]",Women,https://image.hm.com/assets/hm/96/da/96daaeeb8...,193.95
3,Cotton poplin shirt,Long-sleeved shirt in crisp cotton poplin with...,Blue/Striped,Woven,"[{'name': 'Cotton', 'percentage': '100.00'}]","{'value': 22.99, 'currency': 'GBP'}","[M, L, XS, S, XL, XXL, XXS]",Women,https://image.hm.com/assets/hm/9c/40/9c405e26c...,111.50
4,Scuba sweatshirt,Loose-fit sweatshirt in scuba fabric with a st...,Grey marl,Knitted,"[{'name': 'Elastane', 'percentage': '8.00'}, {...","{'value': 27.99, 'currency': 'GBP'}","[XS, M, S, L]",Women,https://image.hm.com/assets/hm/81/18/81186b3f2...,135.75


### Save Data

In [52]:
products_df.to_csv(os.path.join('..', 'data', 'products_raw.csv'), index=False)

In [57]:
with open(os.path.join('..', 'data', 'unique_values.json'), 'w') as json_file:
    json.dump({
        "construction": construction_uniques.tolist(),
        "materials": materials_uniques.tolist(),
        "sizes": sizes_uniques.tolist(),
        "sex": sex_uniques.tolist(),
    },
    json_file,
    indent=4)